# Task 1.1 Baseline: Base Model on GSM8K

In [1]:
import re
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# System prompt: elicit step-by-step reasoning and a consistent final-answer marker.
# We explicitly ask for "Answer: <number>" and discourage \boxed{} (base model often uses it).

SYSTEM_PROMPT = """
You are a math tutor. 
Solve the problem inside <reasoning> tags, then provide the final integer after 'Answer: '.

Example:

Problem: 12 + 7

<reasoning>
1. Start with 12.
2. Add 7.
3. 12 + 7 = 19.
<reasoning>

Answer: 19

It is CRUCIAL that you follow the format EXACTLY. The answer must be provided in the format "Answer: <integer>" ONLY. 
"""

# When solving a problem:
# 1. Show your reasoning step-by-step.
# 2. Use clear numbered steps.
# 3. The final line MUST be exactly in this format:

# Answer: <integer>

# Do not write anything after the final answer.

# Examples:

# Problem: 12 + 7
# Steps:
# 1. Start with 12.
# 2. Add 7.
# 3. 12 + 7 = 19.

# Answer: 19


# Problem: A box has 8 apples. You buy 5 more apples. How many apples do you have?
# Steps:
# 1. Initial apples = 8.
# 2. Apples bought = 5.
# 3. Total apples = 8 + 5.
# 4. 8 + 5 = 13.

# Answer: 13"""

In [2]:
# ── Answer extraction ──
# Primary: "Answer: <integer>". Fallback: \boxed{...} (model often uses it despite prompt).

def extract_boxed(text):
    """Extract content from the last \\boxed{...}, handling nested braces."""
    if not text or "\\boxed{" not in text:
        return None
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    start += len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    if depth == 0:
        return text[start : i - 1].strip()
    return None


def extract_ground_truth(raw_answer):
    """Extract ground-truth answer from GSM8K answer field (ends with #### <number>)."""
    m = re.search(r"####\s*(.+)", raw_answer)
    if m:
        return normalize_answer(m.group(1).strip())
    return normalize_answer(raw_answer.strip())


def normalize_answer(s):
    """Strip and remove commas for numeric comparison."""
    return s.strip().replace(",", "").strip()


def answers_match(pred, gt):
    """Compare predicted and ground-truth answers (both normalized). GSM8K answers are integers."""
    if pred is None or gt is None:
        return False
    pred, gt = normalize_answer(pred), normalize_answer(gt)
    try:
        return int(float(pred)) == int(float(gt))
    except ValueError:
        return pred == gt


def extract_model_answer(text):
    """Extract final answer: prefer 'Answer: <integer>', fallback to \\boxed{...}."""
    if not text:
        return None
    matches = list(re.finditer(r"Answer:\s*([^\s\n]+)", text, re.IGNORECASE))
    if matches:
        return normalize_answer(matches[-1].group(1))
    boxed = extract_boxed(text)
    if boxed is not None:
        return normalize_answer(boxed)
    return None

**Why does the model sometimes use `\boxed{}` instead of "Answer:"?**  
The base model (Qwen2.5-1.5B-Instruct) was trained on a lot of math and reasoning data where final answers are often written as `\boxed{...}` (e.g. from competition math and papers). That format is a strong prior, so the model may use it even when the system prompt asks for "Answer: &lt;number&gt;". We still prefer "Answer:" in the prompt for consistency, but the extractor falls back to `\boxed{...}` so we don't count correct answers as wrong when the model uses that format.

In [3]:
# Load GSM8K test set and take first 100 questions (fixed subset for all experiments)
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
num_test = 100
test_questions = [gsm8k[i]["question"] for i in range(num_test)]
test_ground_truth = [extract_ground_truth(gsm8k[i]["answer"]) for i in range(num_test)]
print(f"Loaded {num_test} test questions. Example GT: {test_ground_truth[0]!r}")

Loaded 100 test questions. Example GT: '18'


In [4]:
# ── Model loading and batched generation ──

def load_model(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    model.eval()
    print(f"Loaded: {model_name}")
    return model, tokenizer


def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions, max_new_tokens=2048):
    """Generate responses for a batch of questions."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses

In [5]:
def evaluate_gsm8k(model, tokenizer, questions, ground_truths, batch_size=16):
    """Zero-shot eval on GSM8K. Returns accuracy and list of (pred, gt, correct, raw_response)."""
    correct = 0
    records = []
    num_samples = len(questions)
    for start in tqdm(range(0, num_samples, batch_size), desc="Evaluating"):
        end = min(start + batch_size, num_samples)
        batch_q = questions[start:end]
        batch_gt = ground_truths[start:end]
        responses = generate_batch(model, tokenizer, batch_q)
        for i, (resp, gt) in enumerate(zip(responses, batch_gt)):
            pred = extract_model_answer(resp)
            is_correct = answers_match(pred, gt)
            if is_correct:
                correct += 1
            records.append({"pred": pred, "gt": gt, "correct": is_correct, "response": resp})
    accuracy = correct / num_samples
    return accuracy, records

In [6]:
# Run baseline evaluation (QUESTION 1)
model, tokenizer = load_model()
accuracy, records = evaluate_gsm8k(
    model, tokenizer, test_questions, test_ground_truth, batch_size=16
)
print(f"\nBase model accuracy on 100 GSM8K test questions: {accuracy:.2%}")
print(f"Correct: {sum(r['correct'] for r in records)} / {len(records)}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


Evaluating: 100%|██████████| 7/7 [00:31<00:00,  4.51s/it]


Base model accuracy on 100 GSM8K test questions: 38.00%
Correct: 38 / 100


## QUESTION 2: Inspect 3 Incorrect Cases

In [7]:
# Get indices of incorrect predictions (run after Q1 evaluation)
incorrect_indices = [i for i, r in enumerate(records) if not r["correct"]]
print(f"Total incorrect: {len(incorrect_indices)}. Inspecting first 3.\n")

# Show 3 failure cases
num_inspect = min(3, len(incorrect_indices))
for case_num, idx in enumerate(incorrect_indices[:num_inspect], 1):
    q = test_questions[idx]
    r = records[idx]
    print("=" * 80)
    print(f"FAILURE CASE {case_num}")
    print("=" * 80)
    print("\nQuestion:")
    print(q)
    print("\nModel's solution (excerpt):")
    # Show first ~600 chars; full response available as r["response"] if needed
    excerpt = r["response"]
    print(excerpt)
    print("\nExtracted answer:", repr(r["pred"]))
    print("Ground-truth answer:", repr(r["gt"]))
    print()

Total incorrect: 62. Inspecting first 3.

FAILURE CASE 1

Question:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Model's solution (excerpt):
<reasoning>
1. Calculate the total cost of buying the house including repairs:
   Total Cost = House Price + Repairs
   Total Cost = $80,000 + $50,000 = $130,000
   
2. Determine the increase in value due to the renovations:
   Increase in Value = Original House Price * Percentage Increase
   Increase in Value = $80,000 * 150% = $80,000 * 1.5 = $120,000
   
3. Calculate the new market value of the house:
   New Market Value = Original House Price + Increase in Value
   New Market Value = $80,000 + $120,000 = $200,000
   
4. Calculate the profit made from selling the house:
   Profit = New Market Value - Total Cost
   Profit = $200,000 - $130,000 = $170,000
</reasoning>

Answer: 170000

Extracted answer: '170000'

### Failure mode classification and patterns

**Case 1 – Failure mode:** Arithmetic slip  

**Case 2 – Failure mode:** Arithmetic slip  

**Case 3 – Failure mode:** Misunderstanding the problem (and reasoning error)  

**Recurring patterns:** arithmetic slips or misunderstanding the problem

## QUESTION 3: LoRA Hyperparameters 

### 1. LoRA rank (r)

**What it controls:** LoRA adapts a weight matrix $W$ by a low-rank update $W' = W + B A$, where $B$ is $(d \times r)$ and $A$ is $(r \times k)$. The rank $r$ directly controls the capacity of the adapter: how many independent "directions" the model can use to adapt each layer.

**If you increase $r$:**  
- **Pros:** More capacity to learn task-specific patterns; can improve accuracy and fit harder tasks.  
- **Cons:** higher memory compute per step; longer training; higher overfitting risk.

**If you decrease $r$:**  
- **Pros:** lower memory and compute; faster steps; less overfitting. 
- **Cons:** may underfit on complex tasks; model might not be able to capture all needed adaptations.

---

### 2. LoRA alpha

**What it controls:** The LoRA forward pass is typically $y = Wx + (\alpha/r)\, (BA)x$. So the adapter's output is scaled by $\alpha/r$. For a fixed $r$, $\alpha$ controls the influence of the adapter relative to the frozen base model. 

**If you increase $\alpha$:**  
- **Pros:** Stronger effective learning signal from the adapter; can speed up adaptation and make the fine-tuned behavior more pronounced.  
- **Cons:** Can make training less stable (larger effective step size); in extreme cases the adapter can dominate and hurt generalization.

**If you decrease $\alpha$:**  
- **Pros:** **More stable** training; adapter changes are gentler.  
- **Cons:** **slower** adaptation; may need more epochs or risk underfitting if the model barely changes.

---

### 3. Gradient accumulation

**What it controls:** Number of mini-batch forward/backward passes before each optimizer step;

**If you increase gradient accumulation:**  
- **Pros:** Larger effective batch size; more stable gradient estimates; often smoother loss and more predictable training;
- **Cons:** more time and compute; can generalize slightly worse.

**If you decrease gradient accumulation:**  
- **Pros:** Fewer passes per step; smaller effective batch.  
- **Cons:** less stable training, may need lower learning rate or more careful tuning